# A Forecasting Procedure

We have several ways to produce a forecast. Now we need a procedure for deciding which one to use. We will work through one planning problem, following steps we can reuse in other forecasting projects.

| Step | Question |
| --- | --- |
| 1. Understand the forecasting problem | What will be predicted, how far ahead, and for what decision? |
| 2. Examine the available history | What patterns should the forecasts account for? |
| 3. Plan a fair comparison | How will we recreate forecasts using the information available at the time? |
| 4. Start with simple forecasts | What performance can we achieve with a baseline? |
| 5. Try a few reasonable models | Which alternatives might improve the forecasts? |
| 6. Evaluate the evidence | How large are the errors, and how do they change with horizon? |
| 7. Choose, forecast, and keep checking | What should we use, and how will we track its performance? |

## References

This procedure draws on [FPP Pythonic Chapter 5, The Forecaster's Toolbox](https://otexts.com/fpppy/05-toolbox.html): the workflow in §5.1, baselines in §5.2, accuracy in §5.8, and time-series cross-validation in §5.10. The case reuses the holiday-trips data from 12b. [Data provenance](../12b-exponential-smoothing/data/README.md) records the Tourism Research Australia source and frozen course extract.

[StatsForecast](https://nixtlaverse.nixtla.io/statsforecast/src/core/core.html) provides a common interface for the forecasting methods and rolling-origin comparisons. [UtilsForecast](https://nixtlaverse.nixtla.io/utilsforecast/evaluation.html) calculates forecast accuracy. The course environment includes these packages. In Colab, run `%pip install statsforecast==2.0.1` before running the notebook.

## Understand the Forecasting Problem

A tourism planning team prepares a quarterly outlook for domestic holiday travel in Australia. The outlook helps coordinate campaign timing and visitor-service planning. Each quarter, the team receives the latest trip count and updates its forecasts for the next four quarters.

Our question is: **Which forecasting procedure should the team use for that quarterly outlook?**

- **Target:** domestic overnight holiday trips in each quarter, expressed in millions of trips.
- **Forecast horizons:** one, two, three, and four quarters ahead.
- **Update schedule:** issue a fresh set of four forecasts after each quarterly observation arrives.
- **Primary accuracy measure:** MAE, which expresses the average absolute miss in millions of trips. Also report RMSE to give larger misses more weight.

We will give each horizon equal weight in the overall comparison, reflecting the team's use of the complete four-quarter outlook.

## Examine the Available History

Place ourselves at the end of 2009. The available history contains 48 quarters, beginning in 1998. We have seen this holiday-trips series in 12b; now we will measure how well its forecasts perform.

The setup loads the existing course extract and expresses trips in millions. It prepares the three columns used by StatsForecast: `unique_id` identifies the series, `ds` records the date, and `y` contains the observed value. We have one series, labeled `Holiday`.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Naive, SeasonalNaive
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

# Reuse the course extract, with the published copy available for Colab.
candidates = [
    Path("../12b-exponential-smoothing/data/australian-holiday-trips.csv"),
    Path("content/lectures/12b-exponential-smoothing/data/australian-holiday-trips.csv"),
    Path("lectures/12b-exponential-smoothing/data/australian-holiday-trips.csv"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is not None:
    frame = pd.read_csv(data_path)
else:
    data_url = ("https://raw.githubusercontent.com/olearydj/INSY7130/main/"
                "lectures/12b-exponential-smoothing/data/australian-holiday-trips.csv")
    with urlopen(data_url, timeout=30) as response:
        frame = pd.read_csv(response)

# Convert quarterly labels to quarter-start dates and thousands of trips to millions.
quarters = pd.PeriodIndex(frame["Quarter"].str.replace(" ", ""), freq="Q")
data = pd.DataFrame({"unique_id": "Holiday", "ds": quarters.to_timestamp(),
                     "y": frame["Trips"] / 1000})
assert data["ds"].is_unique and data["y"].notna().all()
assert quarters.equals(pd.period_range("1998Q1", "2017Q4", freq="Q"))
initial_history = data.loc[data["ds"] < "2010-01-01"].copy()
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
display(initial_history.head())

`y` is measured in millions of trips. Each `ds` label marks the start of the quarter represented by that row. The initial history ends in 2009 Q4.

Look at the time plot, then align the years by quarter. The second panel helps us see whether the same seasonal positions tend to be high or low.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(initial_history["ds"], initial_history["y"])
axes[0].set(title="History Available at the End of 2009", xlabel="Year",
            ylabel="Holiday trips (millions)")
# Align each year's four observations to expose the recurring seasonal pattern.
for _year, annual in initial_history.groupby(initial_history["ds"].dt.year):
    axes[1].plot(annual["ds"].dt.quarter, annual["y"], alpha=0.5)
axes[1].set(title="Each Line Is One Year", xlabel="Quarter",
            ylabel="Holiday trips (millions)", xticks=[1, 2, 3, 4])
plt.tight_layout()
plt.show()
plt.close(fig)

The first quarter tends to have the most trips. Counts vary between years, while the recurring quarterly pattern gives us a reason to use forecasts that preserve seasonality.

## Plan a Fair Comparison

Recreate the team's task at historical forecast origins. A **forecast origin** is the last period whose observation is available when we issue a forecast. At each origin, use the available history to forecast the following four quarters. Then move forward one quarter and repeat.

Set the periods before comparing methods:

| Period | Role |
| --- | --- |
| 1998–2009 | Initial history for the first forecast |
| 2010–2015 | Development outcomes used to compare and choose procedures |
| 2016–2017 | Reserved later outcomes used to check the selected procedure |

The development forecasts begin at the end of 2009 and continue through the end of 2014. The last origin forecasts all four quarters of 2015. We have 21 origins, each with four horizons, yielding 84 forecast errors per method.

Retain all observations through 2015 for this development comparison.

In [ ]:
horizon = 4
development = data.loc[data["ds"] < "2016-01-01"].copy()

The date filter establishes which outcomes can inform our choice of procedure. StatsForecast will recreate the shorter history available at each origin within this period.

The diagram shows the first three origins. Each row gains one observed quarter and issues a fresh four-quarter forecast.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for row, stop in enumerate([48, 49, 50]):
    ax.scatter(np.arange(stop), np.repeat(row, stop), color="C0", s=14)
    ax.scatter(np.arange(stop, stop + horizon), np.repeat(row, horizon),
               color="C1", marker="s", s=35)
ax.set(xlim=(40, 54), ylim=(-0.6, 2.6),
       yticks=[0, 1, 2], yticklabels=["Origin: 2009 Q4", "Origin: 2010 Q1",
                                    "Origin: 2010 Q2"],
       xticks=[40, 44, 48, 52], xticklabels=["2008 Q1", "2009 Q1", "2010 Q1", "2011 Q1"],
       title="Rolling Origins: Circles Are History; Squares Are Forecast Targets")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
plt.close(fig)

This is **rolling-origin evaluation**, also called time-series cross-validation. The expanding history lets each new observation become available for the following forecast. We will refit the fitted methods at every origin.

A target quarter can be predicted at several origins. For example, a four-quarter-ahead prediction and a one-quarter-ahead prediction for the same target use different histories. We keep the horizon when comparing their errors.

## Start With Simple Forecasts

The naive forecast repeats the latest observation for all future quarters. Seasonal-naive repeats the last observed yearly cycle, matching each future quarter with the same quarter a year earlier.

At the end of 2009, seasonal-naive uses 2009 Q1 to predict 2010 Q1, 2009 Q2 to predict 2010 Q2, and so on. Construct that forecast directly once so we can see what the baseline does.

In [ ]:
# Repeat the latest complete year's four quarterly observations.
first_targets = pd.date_range("2010-01-01", periods=horizon, freq="QS")
seasonal_baseline = pd.Series(
    initial_history["y"].tail(4).to_numpy(), index=first_targets.to_period("Q"),
    name="Seasonal-naive forecast (millions)",
)
display(seasonal_baseline.to_frame())

`.tail(4)` retrieves the last four quarterly counts, in seasonal order. Assigning those values to the following four quarters produces the forecast: about 10.132 million trips in Q1, then 8.400, 8.690, and 8.262 million. A fitted model will need to improve on this baseline.

## Try a Few Reasonable Models

Compare four procedures:

| Procedure | What It Uses |
| --- | --- |
| Naive | The latest observed value |
| Seasonal-naive | The latest observation from the same quarter of the year |
| AutoETS | Automatically selected exponential-smoothing components and fitted parameters |
| AutoARIMA | Automatically selected ARIMA orders and fitted coefficients |

AutoETS selects an exponential-smoothing specification, using the level, trend, and seasonal ideas developed in 12b. Both automatic procedures make their selections from the history supplied at each origin.

Give the four models to one `StatsForecast` object, then issue their forecasts from the initial history.

In [ ]:
models = [
    Naive(),
    SeasonalNaive(season_length=4),
    AutoETS(season_length=4),
    AutoARIMA(season_length=4),
]
model_names = [str(model) for model in models]
forecaster = StatsForecast(models=models, freq="QS")

# Fit every procedure to the same history and forecast the next four quarters.
first_forecasts = forecaster.forecast(df=initial_history, h=horizon)
display(first_forecasts)

`season_length=4` supplies the yearly seasonal period. `freq="QS"` identifies quarterly observations labeled at quarter starts. `.forecast()` fits the models and returns their predictions in one table, with a column for each procedure. `model_names` retains those column names for scoring and plotting.

Reveal the observations for 2010 and compare this first set of forecasts. Look for the seasonal peak and the predicted level across all four quarters.

In [ ]:
fig, ax = plt.subplots()
recent = initial_history.tail(8)
ax.plot(recent["ds"], recent["y"], label="Available history", color="gray")
actual_first = development.loc[development["ds"].isin(first_forecasts["ds"])]
ax.plot(actual_first["ds"], actual_first["y"], label="Observed in 2010",
        color="black", marker="o", linewidth=2)
for name in model_names:
    ax.plot(first_forecasts["ds"], first_forecasts[name],
            label=name, linestyle="--", marker=".")
ax.set(title="Four-Quarter Forecasts Issued at the End of 2009",
       xlabel="Quarter", ylabel="Holiday trips (millions)")
ticks = pd.PeriodIndex(["2008Q1", "2008Q3", "2009Q1", "2009Q3", "2010Q1", "2010Q3"],
                       freq="Q")
ax.set_xticks(ticks.to_timestamp(), ticks.astype(str))
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
plt.close(fig)

The seasonal methods capture the high first quarter. The naive forecast holds the last observed value across the year. This origin supplies four errors per method; next we will repeat the task as the origin moves forward.

## Evaluate the Evidence

Use `StatsForecast` to perform the rolling-origin comparison on the development period. This recreates a sequence of forecasts as though we were moving through the historical data one quarter at a time.

At the first origin, the end of 2009:

1. Give each method the observations from 1998 Q1 through 2009 Q4.
2. Fit the models using that history. AutoETS and AutoARIMA also select their model specifications.
3. Forecast the four quarters of 2010.
4. Retain those predictions alongside the actual 2010 observations for scoring.

Then advance one quarter:

1. Add the observed 2010 Q1 value to the available history.
2. Fit each method again using history through 2010 Q1.
3. Forecast 2010 Q2 through 2011 Q1.
4. Retain that new set of predictions and outcomes.

Repeat through the origin at 2014 Q4, whose forecasts cover 2015.

That produces 21 origins × 4 horizons = 84 predictions per method. Each prediction uses only observations available at its origin. We can then compare methods across those forecasts, both overall and separately for each horizon.

In [ ]:
cv = forecaster.cross_validation(
    df=development,
    h=4,
    n_windows=21,
    step_size=1,
    refit=True,
)
display(cv.head(8))

`h` sets the number of future quarters; `n_windows` sets the number of origins. `step_size=1` advances by one quarter. `refit=True` fits each procedure again using the history available at that origin, including automatic selection for AutoETS and AutoARIMA.

In the returned table, `cutoff` identifies the forecast origin, `ds` identifies the target quarter, and `y` is its observed outcome. Each model column contains that procedure's prediction. The first four rows belong to the first origin; the next four belong to the following origin.

### Compare Typical and Large Errors

MAE averages absolute errors. RMSE averages squared errors and takes a square root, giving larger misses greater influence. Both are measured in millions of trips.

Use the standard accuracy functions to score all 84 development forecasts per method, then arrange the results by MAE. This is average cross-validation / rolling-origin error.

In [ ]:
# Pool the forecast errors across origins and horizons for the overall scores.
accuracy = evaluate(cv.drop(columns="cutoff"), models=model_names, metrics=[mae, rmse])
scores = accuracy.set_index("metric")[model_names].T.sort_values("mae")
selected_name = scores.index[0]
display(scores)
print("Selected procedure:", selected_name)

`evaluate()` calculates the requested metrics from observed `y` values and the model predictions. Removing `cutoff` from this scoring view pools all origins before calculating each metric. `.T` turns the metric rows into columns so each procedure occupies one row. Sorting by `mae` puts our preferred procedure first.

Seasonal-naive has the lowest MAE, about 0.391 million trips, and the lowest RMSE, about 0.464 million. AutoETS has MAE about 0.424 million; AutoARIMA about 0.508 million. Preserving last year's seasonal cycle works well across these development forecasts.

### Examine the Horizon

The team uses all four quarters of the outlook. An overall score can conceal a procedure that performs well nearby and poorly farther ahead. Calculate MAE separately for each horizon.

In [ ]:
# Number the four targets within each origin: one through four quarters ahead.
cv["horizon"] = cv.groupby("cutoff").cumcount() + 1
horizon_accuracy = evaluate(
    cv.drop(columns="cutoff"), models=model_names, metrics=[mae], id_col="horizon",
)
display(horizon_accuracy)

`.cumcount()` counts rows within each origin, starting at zero; adding one supplies horizons 1–4. Using `id_col="horizon"` tells the evaluator to score each horizon separately. Each row now summarizes 21 forecasts with the same lead time.

Plot those values to compare accuracy as we look farther ahead.

In [ ]:
ax = horizon_accuracy.set_index("horizon")[model_names].plot(marker="o")
ax.set(title="Development MAE by Forecast Horizon", xlabel="Quarters ahead",
       ylabel="MAE (millions of trips)", xticks=[1, 2, 3, 4])
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
plt.close(ax.figure)

AutoETS has the lowest next-quarter MAE, about 0.352 million trips, followed by seasonal-naive at 0.362 million. Seasonal-naive performs better at the longer horizons and leads the complete four-quarter comparison. The horizon we care about affects the choice.

The naive forecast improves at horizon four because its latest observation comes from the same seasonal position as that target. At this horizon, naive and seasonal-naive issue identical predictions.

Which procedure would you choose for a team that needs only the next quarter? What would you choose for our team's four-quarter outlook?

##### Answer

AutoETS leads the next-quarter comparison by about 0.010 million trips, or 10,000 trips. Seasonal-naive leads when we give all four horizons equal weight. Our stated planning task therefore selects seasonal-naive.

### Check the Selected Procedure on Later Outcomes

We chose the procedure using outcomes through 2015. Now check that selected procedure against the reserved 2016–2017 outcomes. Start at the end of 2015 and advance through the end of 2016, issuing four-quarter forecasts at five origins.

Create a forecaster containing the selected model and evaluate it over those later origins.

In [ ]:
selected_model = models[model_names.index(selected_name)]
chosen = StatsForecast(models=[selected_model], freq="QS")
final_cv = chosen.cross_validation(df=data, h=4, n_windows=5, step_size=1, refit=True)
final_accuracy = evaluate(
    final_cv.drop(columns="cutoff"), models=[selected_name], metrics=[mae, rmse],
)
display(final_accuracy)

`model_names.index(selected_name)` locates the chosen procedure in the original model list. The five origins and four horizons produce twenty forecasts, all targeting quarters in 2016–2017. Each origin uses the expanding history available at that point.

Seasonal-naive has MAE about 0.365 million trips and RMSE about 0.464 million on these later forecasts. Its absolute accuracy remains similar to the development result.

Also check the direction of its errors. With error defined as actual minus predicted trips, a positive mean indicates underprediction.

In [ ]:
# Keep the error sign: positive values mean actual travel exceeded the forecast.
final_errors = final_cv["y"] - final_cv[selected_name]
print(f"Mean forecast error: {final_errors.mean():.3f} million trips")

Subtracting the selected forecast column from `y` calculates the twenty signed errors. Their mean is about 0.291 million trips: travel exceeded the forecast on average. We will track that tendency as new outcomes arrive.

## Choose, Forecast, and Keep Checking

Recommend seasonal-naive for the team's four-quarter outlook. Its seasonal structure matches the recurring pattern, its development errors were lowest across the four horizons, and its later-period accuracy supports using it.

At the end of 2017, issue the next four-quarter outlook using all observations now available.

In [ ]:
future_forecast = chosen.forecast(df=data, h=4)
display(future_forecast)

`.forecast()` applies the selected procedure to the complete available history and creates the next four quarterly dates. The outlook predicts about 12.406 million trips in 2018 Q1, followed by 10.471, 10.499, and 11.211 million. Seasonal-naive obtains these values by repeating the corresponding quarters of 2017.

Each quarter, retain the forecasts when they are issued. When the actual trip count arrives, calculate its errors at the horizons where it was forecast. Track absolute errors and mean error over time, with particular attention to persistent underprediction. Use that evidence to decide when to revisit the candidate comparison.

## The Procedure to Reuse

1. State the target, decision, horizons, and update schedule.
2. Inspect the history available at the first forecast origin.
3. Set the evaluation periods and primary accuracy measure.
4. Establish simple baselines.
5. Fit a small set of plausible alternatives using the same histories.
6. Compare forecast errors across origins and horizons; check the selected procedure on later outcomes.
7. Issue the forecast and keep a record that supports ongoing checking.

For this case, that procedure led us to seasonal-naive. For another series or forecasting task, we can follow the same steps and reach a different choice.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).